In [44]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model(model_name)
    
    def _load_model(self, model_name: str):
        self.model = SentenceTransformer(model_name)
        print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
    
    def embed_documents(self, texts: List[str]) -> np.ndarray:
        """Embed a list of documents and return their embeddings."""
        return self.model.encode(texts, convert_to_numpy=True)

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Backward-compatible alias for earlier notebook cells."""
        return self.embed_documents(texts)
    
    def embed_query(self, queries: List[str]) -> np.ndarray:
        """Embed a list of query strings and return their embeddings."""
        return self.model.encode(queries, convert_to_numpy=True)


## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager.model

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6265.15it/s]


Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_697234/1956127999.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [30]:
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def process_all_pdf(pdf_documents):
    """ Process all PDFs in a directory"""
    all_documents =[]
    pdf_dir= Path(pdf_documents)
    Path(pdf_documents)
    pdf_files= list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files in {pdf_documents}")
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {pdf_file}")
        except Exception as e:
            print(f"Error loading {pdf_file}: {e}")
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the directory
all_documents = process_all_pdf("data/pdf/")

        

Found 2 PDF files in data/pdf/
Processing data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf
Loaded 401 documents from data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf
Processing data/pdf/O'Reilly - Python Cookbook.pdf
Loaded 677 documents from data/pdf/O'Reilly - Python Cookbook.pdf
Total documents loaded: 1078


In [32]:
all_documents

[Document(metadata={'producer': 'calibre 7.4.0', 'creator': 'calibre 7.4.0', 'creationdate': '2025-12-24T00:08:07+00:00', 'source': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'file_path': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'total_pages': 401, 'format': 'PDF 1.4', 'title': '946277975', 'author': 'Unknown', 'subject': '', 'keywords': '', 'moddate': '2025-12-24T00:08:07+00:00', 'trapped': '', 'modDate': "D:20251224000807+00'00'", 'creationDate': "D:20251224000807+00'00'", 'page': 0}, page_content=''),
 Document(metadata={'producer': 'calibre 7.4.0', 'creator': 'calibre 7.4.0', 'creationdate': '2025-12-24T00:08:07+00:00', 'source': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'file_path': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'total_pages': 401, 'format': 'PDF 1.4', 'title': '946277975', 'author': 'Unknown', 'subject': '', 'keywords': '', 'm

In [33]:
# Text splitting into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """ Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len, separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs



In [34]:
class ChromaVectorStore:
    def __init__(self, collection_name: str = "test_collection"):
        self.client = chromadb.PersistentClient(path="./chroma_db")
        self.collection = self.client.get_or_create_collection(name=collection_name, embedding_function=None)
        self.id = str(uuid.uuid4())
    
    def add_documents(self, documents, embeddings):
        """Add documents and their embeddings to the vector store."""
        try:
            doc_ids = []
            doc_contents = []
            doc_metadatas = []
            embedding_list = []

            if embeddings is None:
                raise ValueError("Embeddings cannot be None")

            if isinstance(embeddings, np.ndarray):
                embedding_values = embeddings
            else:
                embedding_values = np.asarray(embeddings)

            if len(documents) != len(embedding_values):
                raise ValueError(
                    f"Document count ({len(documents)}) does not match embedding count ({len(embedding_values)})"
                )

            for i, (doc, embedding) in enumerate(zip(documents, embedding_values)):
                doc_id = f"{self.id}_{i}"
                doc_ids.append(doc_id)

                if hasattr(doc, 'page_content') and hasattr(doc, 'metadata'):
                    doc_contents.append(doc.page_content)
                    doc_metadatas.append(doc.metadata)
                else:
                    doc_contents.append(doc)
                    doc_metadatas.append({"source": "text_input", "index": i})

                if hasattr(embedding, "tolist"):
                    embedding_list.append(embedding.tolist())
                else:
                    embedding_list.append(list(embedding))
            
            self.collection.add(
                ids=doc_ids,
                documents=doc_contents,
                metadatas=doc_metadatas,
                embeddings=embedding_list
            )
            
            print(f"Added {len(doc_ids)} documents to vector store")
            
        except Exception as e:
            print(f"Error preparing documents: {e}")
            raise

In [35]:
# Packages are already available in the current environment.
print('Skipping package install step; dependencies are already available.')

Skipping package install step; dependencies are already available.


In [46]:
class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            self.model = None
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if not self.model:
            raise ValueError("Model is not loaded. Cannot generate embeddings.")
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise

    def embed_documents(self, texts: List[str]) -> np.ndarray:
        """Embed a list of documents and return their embeddings."""
        return self.generate_embeddings(texts)

    def embed_query(self, queries: List[str]) -> np.ndarray:
        """Embed a list of query strings and return their embeddings."""
        if not self.model:
            raise ValueError("Model is not loaded. Cannot embed queries.")
        try:
            print(f"Generating embeddings for {len(queries)} queries...")
            embeddings = self.model.encode(queries, show_progress_bar=False, convert_to_numpy=True)
            return embeddings
        except Exception as e:
            print(f"Error embedding queries: {e}")
            raise
        
    def get_embedding_dimension(self) -> int:
        """Get the dimension of the embeddings."""
        if not self.model:
            raise ValueError("Model is not loaded. Cannot get embedding dimension.")
        return self.model.get_sentence_embedding_dimension()


## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5822.83it/s]


Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_697234/4217247036.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [37]:
### Vector Store
import os
import numpy as np

class VectorStore:
    def __init__(self, collection_nmae: str = "pdf_documents", persist_directory: str = "./data/vector_store"):
        self.collection_name = collection_nmae
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            #Get or create collections
            self.collection = self.client.get_or_create_collection(name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings"})
            print(f"Vector DB initialized successfully. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {len(self.collection.get()['ids'])}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):   
        """Add documents and their embeddings to the collection."""
        if embeddings is None:
            raise ValueError("Embeddings cannot be None.")

        embedding_values = np.asarray(embeddings)
        if len(documents) != len(embedding_values):
            raise ValueError("Number of documents and embeddings must match.")

        print(f"Adding {len(documents)} documents to the vector store...")
        try:
            ids = []
            metadatas = []
            document_texts = []
            embedding_list = []
            for i, (doc, embedding) in enumerate(zip(documents, embedding_values)):
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)

                base_metadata = dict(doc.metadata or {})
                base_metadata.update({
                    "doc_index": i,
                    "content_length": len(doc.page_content),
                })
                metadatas.append(base_metadata)

                document_texts.append(doc.page_content)

                if hasattr(embedding, "tolist"):
                    embedding_list.append(embedding.tolist())
                else:
                    embedding_list.append(list(embedding))
        except Exception as e:
            print(f"Error preparing documents for addition: {e}")
            raise
        
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_texts,
                embeddings=embedding_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Vector DB initialized successfully. Collection: pdf_documents
Existing documents in collection: 2131


In [38]:
## Convert the text to embeddings and store them in the vector store
if 'chunks' not in globals():
    chunks = split_documents(all_documents)

texts = [doc.page_content for doc in chunks]

## Generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

## Store the embeddings and documents in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 2131 texts...


Batches: 100%|██████████| 67/67 [00:06<00:00, 10.11it/s]


Generated embeddings with shape: (2131, 384)
Adding 2131 documents to the vector store...
Successfully added 2131 documents to the vector store.


In [39]:
# In [30]:
for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i}:")
    print(chunk.page_content[:100])
    print(chunk.metadata)
    print("---")


Chunk 0:
OceanofPDF.com
{'producer': 'calibre 7.4.0', 'creator': 'calibre 7.4.0', 'creationdate': '2025-12-24T00:08:07+00:00', 'source': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'file_path': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'total_pages': 401, 'format': 'PDF 1.4', 'title': '946277975', 'author': 'Unknown', 'subject': '', 'keywords': '', 'moddate': '2025-12-24T00:08:07+00:00', 'trapped': '', 'modDate': "D:20251224000807+00'00'", 'creationDate': "D:20251224000807+00'00'", 'page': 2}
---
Chunk 1:
Building LLM Agents with RAG, Knowledge Graphs & Reflection
A Practical Guide to Building Intelligen
{'producer': 'calibre 7.4.0', 'creator': 'calibre 7.4.0', 'creationdate': '2025-12-24T00:08:07+00:00', 'source': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'file_path': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'total_pages': 401, 'format': 'PDF 1

In [40]:
# In [30]:
print(f"Embeddings shape: {embeddings.shape}")
print(f"First embedding sample: {embeddings[0][:5]}...")


Embeddings shape: (2131, 384)
First embedding sample: [-0.01970141 -0.01791877 -0.08208188 -0.00243174  0.12395097]...


In [47]:
class RAGRetriever:
    """ Handles query based retrieval from the vector store using embeddings and cosine similarity. """
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAGRetriever with a vector store and an embedding manager.
        Args:
            vector_store (VectorStore): The vector store instance for document retrieval.
            embedding_manager (EmbeddingManager): The embedding manager for generating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    def retrieve(self, query: str, top_k: int =5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve the top_k most relevant documents for a given query.
        Args:
            query (str): The input query string.
            top_k (int): The number of top documents to retrieve.
            score_threshold (float): Minimum cosine similarity score to consider a document relevant.
        Returns:
            List[Dict[str, Any]]: A list of dictionaries containing the retrieved documents and their metadata.
            
        """
        
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")
        
        
        #Generate query embedding
        query_embedding = self.embedding_manager.embed_query([query])[0]
        
        
        # Search in Vector store
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                
            )
            
            #Process Results
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances =results['distances'][0]
                ids = results['ids'][0]
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to cosine similarity score
                    score = 1 - distance
                    if score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "document": document,
                            "metadata": metadata,
                            "score": score,
                            "distance": distance,
                            "rank": i + 1
                        })
                    print(f"Retrieved doc {i}: ID={doc_id}, Score={score:.4f}, Metadata={metadata}")
            else:
                print("No documents retrieved from the vector store.")
            return retrieved_docs
        except Exception as e:
            print(f"Error querying vector store: {e}")
            raise
rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)
rag_retriever
        
        

In [48]:
rag_retriever.retrieve("What is the main topic of the document?", top_k=3, score_threshold=0.5) 

Retrieving documents for query: 'What is the main topic of the document?'
Top K: 3, Score Threshold: 0.5
Generating embeddings for 1 queries...
Retrieved doc 0: ID=doc_9ebb56ac_159, Score=-0.2806, Metadata={'creator': 'calibre 7.4.0', 'doc_index': 159, 'creationdate': '2025-12-24T00:08:07+00:00', 'subject': '', 'moddate': '2025-12-24T00:08:07+00:00', 'author': 'Unknown', 'content_length': 663, 'title': '946277975', 'trapped': '', 'keywords': '', 'file_path': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'format': 'PDF 1.4', 'producer': 'calibre 7.4.0', 'modDate': "D:20251224000807+00'00'", 'source': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'creationDate': "D:20251224000807+00'00'", 'total_pages': 401, 'page': 134}
Retrieved doc 1: ID=doc_89edb10a_159, Score=-0.2806, Metadata={'page': 134, 'source': 'data/pdf/Building_LLM_Agents_with_RAG_Knowledge_Graphs_-_Mira_S_Devlin.pdf', 'trapped': '', 'format': 'PDF 1.4', 'total_

[]

In [49]:
rag_retriever.retrieve("Sorting a dictionary", top_k=3, score_threshold=0.5) 

Retrieving documents for query: 'Sorting a dictionary'
Top K: 3, Score Threshold: 0.5
Generating embeddings for 1 queries...
Retrieved doc 0: ID=doc_7888c4b9_650, Score=0.4978, Metadata={'modDate': "D:20030322073557+08'00'", 'source': "data/pdf/O'Reilly - Python Cookbook.pdf", 'creationdate': '2003-03-22T07:15:58+00:00', 'content_length': 587, 'creator': 'Chapter 3 -20.doc - Microsoft Word', 'producer': 'Acrobat PDFWriter 5.0 for Windows NT', 'subject': '', 'author': 'kefoo', 'format': 'PDF 1.4', 'page': 77, 'file_path': "data/pdf/O'Reilly - Python Cookbook.pdf", 'moddate': '2003-03-22T07:35:57+08:00', 'total_pages': 677, 'creationDate': 'D:20030322071558Z', 'title': 'Chapter 3 -20.doc', 'trapped': '', 'keywords': '', 'doc_index': 650}
Retrieved doc 1: ID=doc_56d891a3_650, Score=0.4978, Metadata={'keywords': '', 'doc_index': 650, 'creationDate': 'D:20030322071558Z', 'format': 'PDF 1.4', 'page': 77, 'modDate': "D:20030322073557+08'00'", 'content_length': 587, 'trapped': '', 'total_pages

[]